# Lesson 15 — scalar index · index บน column ธรรมดา

vector index ช่วย `search(vec)` FTS index ช่วย `search(text)`
แต่ `where("topic = 'memory' AND ts > ...")` ก็มี index ของตัวเอง
**BITMAP** สำหรับ column ที่ค่าซ้ำเยอะ (topic มี 3 ค่า) · **BTREE** สำหรับช่วง (ts เป็นตัวเลข)
11 แถว index ไม่ได้ช่วยให้เร็วขึ้นหรอก บทนี้ดูว่า query plan เปลี่ยนยังไง

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, urllib.request, pathlib
if not pathlib.Path("../data/lesson_data.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Soul-Brews-Studio/lancedb-oracle/main/lessons/data/lesson_data.py", "lesson_data.py")
sys.path.insert(0, "../data")
from lesson_data import load

import datetime as dt
import lancedb
from lancedb.index import BTree, Bitmap

# ts = วินาทีตั้งแต่ 1970 ทำจาก date เพื่อให้เปรียบเทียบเป็นตัวเลขได้
rows = [dict(p, ts=int(dt.datetime.fromisoformat(p["date"]).timestamp())) for p in load("nat_posts.jsonl")]
db = lancedb.connect("./data")
tbl = db.create_table("posts", data=rows, mode="overwrite")
tbl.to_pandas()[["id", "date", "ts", "topic"]]

[2026-09-10T11:36:27Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/15-scalar-index/data/posts.lance, it will be created


,id,date,ts,topic
0,p01,2026-08-28,1787850000,memory
1,p02,2026-08-20,1787158800,memory
2,p03,2026-08-15,1786726800,memory
3,p04,2026-07-31,1785430800,memory
4,p05,2026-06-22,1782061200,memory
5,p06,2026-08-20,1787158800,agents
6,p07,2026-06-17,1781629200,agents
7,p08,2026-05-20,1779210000,agents
8,p09,2026-05-31,1780160400,hardware
9,p10,2026-05-30,1780074000,hardware


**ก่อนมี index** — ดู plan ของ query ที่กรอง topic กับช่วงเวลา
`explain_plan()` คืน string บอกว่า Lance จะทำอะไร ยังไม่รัน
บรรทัดล่างสุดคือ `LanceRead ... full_filter=...` = อ่าน column topic กับ ts ทุกแถว แล้วค่อยกรองในตัว

In [3]:
CUTOFF = int(dt.datetime(2026, 7, 1).timestamp())
q = lambda: tbl.search().where(f"topic = 'memory' AND ts > {CUTOFF}").select(["id", "date", "topic"]).limit(10)
print(q().explain_plan())
q().to_pandas()

ProjectionExec: expr=[id@3 as id, date@4 as date, topic@0 as topic]
  LanceRead: uri=opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/15-scalar-index/data/posts.lance/data, projection=[id, date], source=stream(_rowid)
    LanceRead: uri=opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/15-scalar-index/data/posts.lance/data, projection=[topic, ts], num_fragments=1, range_before=None, range_after=Some(0..10), row_id=true, row_addr=false, full_filter=topic = Utf8("memory") AND ts > Int64(1782838800), refine_filter=topic = Utf8("memory") AND ts > Int64(1782838800)



,id,date,topic
0,p01,2026-08-28,memory
1,p02,2026-08-20,memory
2,p03,2026-08-15,memory
3,p04,2026-07-31,memory


**สร้าง index สองอัน**
API ปัจจุบันคือ `create_index(col, config=Bitmap())` / `BTree()`
`create_scalar_index(col, index_type="BITMAP")` ยังเรียกได้แต่ deprecated ตั้งแต่ 0.25 เหมือน FTS ในบทที่ 8

In [4]:
tbl.create_index("topic", config=Bitmap())
tbl.create_index("ts", config=BTree())
for i in tbl.list_indices():
    print(i.name, i.index_type, i.columns)

topic_idx Bitmap ['topic']
ts_idx BTree ['ts']


**หลังมี index** — query เดิม plan เปลี่ยน
บรรทัดล่างสุดกลายเป็น `ScalarIndexQuery: AND([topic = memory]@topic_idx(Bitmap), [ts > ...]@ts_idx(BTree))`
Lance ถาม index ทั้งสองก่อนว่าแถวไหนตรง แล้ว `LanceRead` ข้างบนถึงอ่านเฉพาะแถวนั้น `refine_filter=--` คือไม่ต้องกรองซ้ำ
ผลลัพธ์เหมือนเดิมทุกแถว ต่างกันแค่วิธีหา

In [5]:
print(q().explain_plan())
q().to_pandas()

LanceRead: uri=opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/15-scalar-index/data/posts.lance/data, projection=[id, date, topic], num_fragments=1, range_before=None, range_after=Some(0..10), row_id=false, row_addr=false, full_filter=topic = Utf8("memory") AND ts > Int64(1782838800), refine_filter=--
  ScalarIndexQuery: query=AND([topic = memory]@topic_idx(Bitmap),[ts > 1782838800]@ts_idx(BTree))



,id,date,topic
0,p01,2026-08-28,memory
1,p02,2026-08-20,memory
2,p03,2026-08-15,memory
3,p04,2026-07-31,memory


**ทำไม 11 แถวถึงไม่ควรทำ** — index คือไฟล์เพิ่ม อ่านเพิ่ม
บน disk แต่ละ index มี directory ของตัวเองใน `_indices/` สองอันรวมกันเกินครึ่งของ data ทั้งตาราง
จุดคุ้มทุนอยู่ที่หลักหมื่นแถวขึ้นไป หรือเมื่อ `where` ตัดแถวออกได้เกิน 90%
ความจำ agent ที่กรองด้วย `agent_id` / `session` ทุกครั้ง ถึงตรงนั้นเร็ว

In [6]:
from pathlib import Path
data_bytes = sum(f.stat().st_size for f in Path("data/posts.lance/data").glob("*.lance"))
print(f"data fragments  {data_bytes:>7} bytes")
for d in sorted(Path("data/posts.lance/_indices").iterdir()):
    n = sum(f.stat().st_size for f in d.rglob("*") if f.is_file())
    print(f"index {d.name[:8]}…  {n:>7} bytes")

data fragments     5220 bytes
index 265a41e7…      812 bytes
index 328e17eb…     2017 bytes
